# RAG Evaluation — facet_embed (Kaggle)

RAG-metrics-only notebook for the **facet_embed** retrieval strategy.
Skips prediction entirely — runs Steps 1→2→6→7→8→9→10 only.

**Requires** (add as Kaggle Dataset input):
- `model_x_ocean` repo cloned to `/kaggle/working/model_x_ocean/`
- `data/vector_db/essays_dual/` index (vectors.faiss + vectors_meta.jsonl)
- `data/profile_db/essays_test/` test profiles (label-blind, already built)
- `data/split/essays/test.csv`

## Setup — clone repo & install dependencies

In [ ]:
!git clone https://github.com/mtrung12/model_x_ocean.git /kaggle/working/model_x_ocean
%cd /kaggle/working/model_x_ocean/
!pip install -q faiss-cpu sentence-transformers einops
!pip install -q -r requirements.txt

## Configuration

In [ ]:
from pathlib import Path
import sys

project_root = Path("/kaggle/working/model_x_ocean")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ── RAG mode ──────────────────────────────────────────────────────────────────
rag_mode = "facet_embed"

# ── RAG metric k values ───────────────────────────────────────────────────────
K_VALUES = [3, 5]

# ── paths ─────────────────────────────────────────────────────────────────────
test_csv        = str(project_root / "data/split/essays/test.csv")
test_profile_db = str(project_root / "data/profile_db/essays_test")
vector_db_dir   = str(project_root / "data/vector_db/essays_dual")
rag_metrics_dir = str(project_root / f"result/rag_ablation/{rag_mode}")

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}
TRAIT_NAMES = list(TRAITS.values())
TRAIT_CODES = {v: k for k, v in TRAITS.items()}
TRAIT_SHORT = {
    "cOPN": "Openness",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

print(f"project_root : {project_root}")
print(f"rag_mode     : {rag_mode}")
print(f"vector_db    : {vector_db_dir}")
print(f"rag_metrics  : {rag_metrics_dir}")

# ── path checks ───────────────────────────────────────────────────────────────
for label, p in [("test_csv", test_csv), ("test_profile_db", test_profile_db), ("vector_db_dir", vector_db_dir)]:
    print(f"  {'OK' if Path(p).exists() else 'MISSING':7s}  {label}: {p}")

## Imports

In [ ]:
import os, json, time
from typing import Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rag.faiss_index      import FAISSIndex
from rag.embedder         import _embed_single, get_embedding, get_dual_embedding
from rag.profiler.store   import ProfileStore
from rag.profiler.prompts import FACETS, slice_profile_for_trait
from rag.facet_vector     import (
    facet_vector, facet_cosine, hybrid_score,
    FACET_ORDER, load_facet_matrix,
)
import rag.retriever as _retriever_mod
from rag.retriever import FeatureRAGRetriever

def normalize_label(val):
    s = str(val).strip().lower()
    if s in ("1", "1.0"): return "high"
    if s in ("0", "0.0"): return "low"
    return s

test_df = pd.read_csv(test_csv)
print(f"Test split: {len(test_df)} rows")

## Step 1 — Load test profiles

Profiles are already built at `data/profile_db/essays_test/` — no API calls needed.

In [ ]:
test_store_path = Path(test_profile_db) / "profile_store.jsonl"
test_store = ProfileStore(str(test_store_path))
test_store.load()

test_profiles_by_idx = {
    int(e["user_id"].split("_")[1]): e
    for e in test_store.get_all() if e.get("valid")
}
print(f"Test profiles loaded: {len(test_profiles_by_idx)}")
if len(test_profiles_by_idx) == 0:
    raise RuntimeError("No test profiles found. Ensure data/profile_db/essays_test/ is in the repo.")

## Step 2 — Retriever adapter

In [ ]:
ALPHA = 0.5

def _profile_to_full_text(entry: Dict) -> str:
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines  = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


class RawpostRetriever(FeatureRAGRetriever):
    """Raw-text embedding query, dense retrieval."""
    def __init__(self, db_dir, test_profiles, text_to_idx):
        super().__init__(db_dir=db_dir)
        self._test_profiles = test_profiles
        self._text_to_idx   = text_to_idx

    def retrieve(self, posts, trait, top_k=5, **kwargs):
        return super().retrieve(posts=posts, trait=trait, top_k=top_k)


class ProfileRetriever(FeatureRAGRetriever):
    """Full 30-facet profile text as query embedding."""
    def __init__(self, db_dir, test_profiles, text_to_idx):
        super().__init__(db_dir=db_dir)
        self._test_profiles = test_profiles
        self._text_to_idx   = text_to_idx

    def _embed_query(self, text, profile_text=None):
        idx   = self._text_to_idx.get(str(text))
        entry = self._test_profiles.get(idx) if idx is not None else None
        prof  = _profile_to_full_text(entry) if (entry and entry.get("valid")) else str(text)
        return super()._embed_query(prof)




class FacetEmbedRetriever(FeatureRAGRetriever):
    """Dual-fused query (raw + full profile), dense-only retrieval (gamma=0)."""
    def __init__(self, db_dir, test_profiles, text_to_idx):
        super().__init__(db_dir=db_dir, use_hybrid=False)
        self._test_profiles = test_profiles
        self._text_to_idx   = text_to_idx

    def _embed_query(self, text, profile_text=None):
        idx   = self._text_to_idx.get(str(text))
        entry = self._test_profiles.get(idx) if idx is not None else None
        if entry and entry.get("valid"):
            prof_text = _profile_to_full_text(entry)
            return get_dual_embedding(str(text), prof_text, alpha=ALPHA)
        return super()._embed_query(str(text))


class HybridFacetRetriever(FeatureRAGRetriever):
    """Dual-fused query + per-trait facet cosine re-ranking."""
    def __init__(self, db_dir, test_profiles, text_to_idx):
        super().__init__(db_dir=db_dir, use_hybrid=True)
        self._test_profiles = test_profiles
        self._text_to_idx   = text_to_idx

    def _embed_query(self, text, profile_text=None):
        idx   = self._text_to_idx.get(str(text))
        entry = self._test_profiles.get(idx) if idx is not None else None
        if entry and entry.get("valid"):
            prof_text = _profile_to_full_text(entry)
            return get_dual_embedding(str(text), prof_text, alpha=ALPHA)
        return super()._embed_query(str(text))

    def _get_query_facet_vector(self, text: str) -> np.ndarray:
        idx   = self._text_to_idx.get(str(text))
        entry = self._test_profiles.get(idx) if idx is not None else None
        if entry and entry.get("valid"):
            return facet_vector(entry, normalize=True)
        return np.zeros(len(FACET_ORDER), dtype=np.float32)

_RETRIEVER_CLASSES = {
    "rawpost":      RawpostRetriever,
    "profile":      ProfileRetriever,
    "facet_embed":  FacetEmbedRetriever,
    "hybrid_facet": HybridFacetRetriever,
}

_OriginalRetriever = _retriever_mod.FeatureRAGRetriever
_text_to_idx = {str(t): i for i, t in enumerate(test_df["text"].tolist())}

def _RetrieverFactory(db_dir=None):
    cls = _RETRIEVER_CLASSES[rag_mode]
    return cls(
        db_dir        = db_dir or vector_db_dir,
        test_profiles = test_profiles_by_idx,
        text_to_idx   = _text_to_idx,
    )

_retriever_mod.FeatureRAGRetriever = _RetrieverFactory
print(f"[adapter] {rag_mode} retriever installed.")

## Step 6 — Compute RAG metrics (MMR@k, HR@k)

In [ ]:
_metric_retriever = _RetrieverFactory(db_dir=vector_db_dir)

rag_rows, per_query_rows = [], []

for k in K_VALUES:
    for tc, trait_full in TRAITS.items():
        match_rates, hits = [], []

        for qi, (_, row) in enumerate(test_df.iterrows()):
            gt = normalize_label(row[tc])
            if gt not in ("high", "low"):
                continue

            retrieved = _metric_retriever.retrieve(
                posts  = str(row["text"]),
                trait  = trait_full,
                top_k  = k,
            )
            n_ret   = len(retrieved)
            matches = sum(
                1 for r in retrieved
                if (r.get("label") or r.get("trait_labels", {}).get(trait_full)) == gt
            )
            rate = matches / n_ret if n_ret > 0 else 0.0
            hit  = int(matches > 0)

            match_rates.append(rate)
            hits.append(hit)
            per_query_rows.append({
                "query_idx": qi, "trait": trait_full, "k": k,
                "true_label": gt, "n_retrieved": n_ret,
                "match_count": matches, "match_rate": rate, "hit": hit,
            })

        mmr = float(np.mean(match_rates))
        hr  = float(np.mean(hits))
        rag_rows.append({
            "rag_mode": rag_mode, "trait": trait_full, "k": k,
            "n_queries": len(match_rates),
            "mean_match_rate": round(mmr, 4),
            "std_match_rate":  round(float(np.std(match_rates)), 4),
            "hit_rate":        round(hr, 4),
        })
        print(f"  k={k}  {trait_full:<28s}  MMR={mmr:.4f}  HR={hr:.4f}")

rag_summary_df = pd.DataFrame(rag_rows)
per_query_df   = pd.DataFrame(per_query_rows)
print(f"\nDone — {len(rag_summary_df)} summary rows")

## Step 7 — Save RAG metrics

In [ ]:
os.makedirs(rag_metrics_dir, exist_ok=True)
rag_summary_df.to_csv(os.path.join(rag_metrics_dir, "rag_summary.csv"), index=False)
per_query_df.to_csv(os.path.join(rag_metrics_dir, "rag_per_query.csv"), index=False)
print(f"Saved to {rag_metrics_dir}/")
print(f"  rag_summary.csv   ({len(rag_summary_df)} rows)")
print(f"  rag_per_query.csv ({len(per_query_df)} rows)")

print("\n=== MMR pivot ===")
display(rag_summary_df.pivot_table(index="trait", columns="k", values="mean_match_rate").round(4))
print("\n=== HR pivot ===")
display(rag_summary_df.pivot_table(index="trait", columns="k", values="hit_rate").round(4))

## Step 8 — Visualisations

In [ ]:
trait_labels = [TRAIT_SHORT[tc] for tc in TRAITS]
x, width = np.arange(len(trait_labels)), 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, k in zip(axes, K_VALUES):
    sub  = rag_summary_df[rag_summary_df["k"] == k].set_index("trait")
    mmrs = [sub.loc[TRAITS[tc], "mean_match_rate"] if TRAITS[tc] in sub.index else 0 for tc in TRAITS]
    hrs  = [sub.loc[TRAITS[tc], "hit_rate"]        if TRAITS[tc] in sub.index else 0 for tc in TRAITS]
    b1 = ax.bar(x - width/2, mmrs, width, label="MMR",      color="#4CAF50", alpha=0.85)
    b2 = ax.bar(x + width/2, hrs,  width, label="Hit-rate", color="#9C27B0", alpha=0.85)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8)
    for bar, v in zip(b1, mmrs):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
    for bar, v in zip(b2, hrs):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(trait_labels, rotation=20, ha="right", fontsize=9)
    ax.set_ylim(0, 1.1); ax.set_title(f"{rag_mode}  k={k}"); ax.legend(fontsize=8); ax.set_ylabel("Score")

plt.suptitle(f"Retrieval quality — {rag_mode} (test split)", fontsize=12)
plt.tight_layout()
plot_path = os.path.join(rag_metrics_dir, "rag_mmr_hr.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

## Step 9 — LaTeX rows

In [ ]:
sep = "=" * 72
print(sep)
print(f"  RAG ABLATION SUMMARY  |  mode={rag_mode}")
print(sep)

for k in K_VALUES:
    sub = rag_summary_df[rag_summary_df["k"] == k].set_index("trait")
    print(f"\n[tab:retrieval-ablation  k={k}]")
    cells = " & ".join(
        f"{sub.loc[TRAITS[tc],'mean_match_rate']:.4f}/{sub.loc[TRAITS[tc],'hit_rate']:.4f}"
        if TRAITS[tc] in sub.index else "?/?"
        for tc in TRAITS
    )
    print(f"  {rag_mode} & {k} & {cells} \\\\")

print(f"\n{sep}")

## Step 10 — Cleanup

In [ ]:
_retriever_mod.FeatureRAGRetriever = _OriginalRetriever
print("[adapter] Original FeatureRAGRetriever restored.")